RSA (аббревиатура от фамилий Rivest, Shamir и Adleman).  
RSA основан на трудности факторизации больших чисел (перемножить два больших простых числа 𝑝 и 𝑞 легко, а разложить их произведение 𝑛 обратно на множители очень трудно).  
Причем существует только одна пара простых множителей 𝑝 и 𝑞 произведение которых даст значение модуля 𝑛.

In [1]:
# Выбрать 2 простых числа p, q
p = 19
q = 41

# Вычисляем модуль n (является частью закрытого и открытого ключа)
n = p * q
print(f"Модуль: {n=}")

Модуль: n=779


In [2]:
# Вычисляем функцию Эйлера φ(n)
phi_n = (p - 1) * (q - 1)
print(f"φ(n)={phi_n}")

φ(n)=720


In [11]:
# Разложим φ(n) на простые множители
def prime_factors(n):
    factors = []
    d = 2
    while d * d <= n:
        while n % d == 0:
            factors.append(d)
            n //= d
        d += 1
    if n > 1:
        factors.append(n)
    return factors


print(set(prime_factors(720)))

{2, 3, 5}


In [4]:
# Разложим φ(n) на простые множители с помощью библиотеки sympy
# %pip install sympy
from sympy import factorint


factors = factorint(phi_n)
print(f"{factors=}")

print(f"φ(n) = {phi_n} = ", end='')
print(" * ".join(f"{key}^{val}" for key, val in factors.items()))

factors={2: 4, 3: 2, 5: 1}
φ(n) = 720 = 2^4 * 3^2 * 5^1


In [ ]:
# Найти 'e', которое 1 < e < φ(n) и НОД(e, φ(n)) = 1
print(f"'e' не должно делится на: {", ".join([f"{key}" for key in factors])}")

e = 0
prime_set = {3,5,7,11,13,17,19,23,29,31,37,41,43,47,53,59,61,67,71,73,79,83,89,97,101,103,107,109,113,127,131,137,139,149,151,157,163}
for i in prime_set:
    if i in factors: continue
    e = i
    break

print(f"Открытая экспонента {e=}")  # В алгоритмах по умолчанию используют e=65_537 (b1_0000_0000_0000_0001)

'e' не должно делится на: 2, 3, 5
Открытая экспонента e=131


In [ ]:
# Перебором подберем такое d, что (d * e) mod φ(n) = 1
d = 0
for i in range(phi_n):
    if (i * e) % phi_n == 1: 
        d = i
        break

print(f"Закрытая экспонента {d=}")

Закрытая экспонента d=11


In [8]:
print(f"🔓 Открытый ключ: ({e=}, {n=})")
print(f"🔒 Закрытый ключ: ({d=}, {n=})")

🔓 Открытый ключ: (e=131, n=779)
🔒 Закрытый ключ: (d=11, n=779)


In [ ]:
# При RSA шифровании сообщение всегда шифруется по модулю n, поэтому сообщение обязано быть меньше n.
# 0 ≤ message < n
message = 123
encrypted = pow(message, e, n)  # Шифруем открытым 🔓 ключем
print(f'Encrypted message: {encrypted}')
decripted = pow(encrypted, d, n)  # Дешифруем закрытым 🔒 ключем 
print(f'Decripted message: {decripted}')

Encrypted message: 738
Decripted message: 123


In [ ]:
# Так же можно поменять ключи местами
message = 123
encrypted = pow(message, d, n)  # Шифруем закрытым 🔒 ключем
print(f'Encrypted message: {encrypted}')
decripted = pow(encrypted, e, n)  # Дешифруем открытым 🔓 ключем
print(f'Decripted message: {decripted}')

Encrypted message: 328
Decripted message: 123


Пример использования RSA  
1. Подпись банковских транзакций.  
Транзакция представляется в виде хэша всех её полей. Эти вычисления могу сделать клиент и банк и они получат одинаковый hash.  
Клиент подписывает этот хэш с помощью своего ключа и отправляет в банк саму транзакцию вместе с подписью.  
Процесс выглядит так:  
клиент вычисляет подпись:  
signature = hash^d mod n  
банк проверяет подпись, выполняя:  
hash = signature^e mod n  
Если в результате проверки получается тот же хэш, значит транзакция была подписана владельцем соответствующего приватного ключа.  

2. SSH-соединение с использованием RSA  
Клиент делает запрос на подключение к серверу.  
Сервер генерирует случайное число x (челлендж) и отправляет его клиенту. Это число не является секретом и доступно всем.  
Клиент подписывает полученное число своим приватным ключом:  
signature = x^d mod n  
и отправляет эту подпись обратно на сервер.  
Сервер проверяет подпись, используя публичный ключ клиента:  
x = signature^e mod n  
Если в результате проверки получается тот же самый челлендж x, это означает, что запрос на соединение действительно исходит от пользователя,
чей публичный ключ был заранее загружен на сервер.  
После успешной проверки сервер разрешает установку SSH-соединения.